# VOC Sedici Load Tables

Notebook de trabajo para replicar el comportamiento actual de `load_vocsedici_tables`.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Setup

In [ ]:
tables = catalog.load("params:vocsedici_load_options.tables")

df_node = catalog.load("raw/voc/node#parquet")
df_node_field_data = catalog.load("raw/voc/node_field_data#parquet")
df_field_nombre = catalog.load("raw/voc/node__field_nombre#parquet")
df_field_apellido = catalog.load("raw/voc/node__field_apellido#parquet")
df_field_orcid = catalog.load("raw/voc/node__field_orcid#parquet")
df_field_mail = catalog.load("raw/voc/node__field_mail#parquet")
df_field_dni = catalog.load("raw/voc/node__field_dni#parquet")
df_field_cuit = catalog.load("raw/voc/node__field_cuit#parquet")
df_field_telefono = catalog.load("raw/voc/node__field_telefono#parquet")
df_field_direccion = catalog.load("raw/voc/node__field_direcci_n#parquet")
df_field_google_scholar = catalog.load("raw/voc/node__field_google_scholar#parquet")
df_field_researchgate = catalog.load("raw/voc/node__field_researchgate#parquet")
df_field_old_id = catalog.load("raw/voc/node__field_old_id#parquet")
df_field_filiacion = catalog.load("raw/voc/node__field_filiacion#parquet")
df_field_nombre_institucion = catalog.load("raw/voc/node__field_nombre_institucion#parquet")
df_field_nombre_institucion_variant = catalog.load("raw/voc/node__field_nombre_institucion_variant#parquet")
df_field_abreviatura = catalog.load("raw/voc/node__field_abreviatura#parquet")
df_field_id_pidu = catalog.load("raw/voc/node__field_id_pidu#parquet")
df_field_id_termino = catalog.load("raw/voc/node__field_id_termino#parquet")
df_field_padre = catalog.load("raw/voc/node__field_padre#parquet")

print(f"tables={len(tables)}")

## Node Logic

In [ ]:
_EXTRACT_META_COLS = [
    "_source_system",
    "_source_table",
    "_extract_datetime",
    "_extract_date",
    "_source_label",
    "_extract_env",
    "_filter_param",
    "_filter_value",
]


def _add_extract_metadata(df: pd.DataFrame) -> pd.DataFrame:
    enriched_df = df.copy()
    for col in _EXTRACT_META_COLS:
        if col not in enriched_df.columns:
            enriched_df[col] = pd.NA
    enriched_df["_extract_datetime"] = pd.to_datetime(
        enriched_df["_extract_datetime"], errors="coerce"
    )
    if "_extract_date" in enriched_df.columns:
        enriched_df["_extract_date"] = pd.to_datetime(
            enriched_df["_extract_date"], errors="coerce"
        ).dt.date
    return enriched_df


def _add_load_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    enriched_df = df.copy()
    if load_datetime is None:
        load_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)
    enriched_df["_load_datetime"] = pd.to_datetime(load_datetime)
    return enriched_df


def _prepare_table(df: pd.DataFrame) -> pd.DataFrame:
    prepared_df = _add_extract_metadata(df).convert_dtypes()
    return _add_load_metadata(prepared_df)


def load_vocsedici_tables(*dataframes):
    return tuple(_prepare_table(df) for df in dataframes)

## Run

In [ ]:
loaded_tables = load_vocsedici_tables(
    df_node,
    df_node_field_data,
    df_field_nombre,
    df_field_apellido,
    df_field_orcid,
    df_field_mail,
    df_field_dni,
    df_field_cuit,
    df_field_telefono,
    df_field_direccion,
    df_field_google_scholar,
    df_field_researchgate,
    df_field_old_id,
    df_field_filiacion,
    df_field_nombre_institucion,
    df_field_nombre_institucion_variant,
    df_field_abreviatura,
    df_field_id_pidu,
    df_field_id_termino,
    df_field_padre,
)

len(loaded_tables)

In [ ]:
table_names = tables

for table_name, df in zip(table_names, loaded_tables):
    print(f"\n=== {table_name} ===")
    print(f"rows={len(df):,} cols={len(df.columns)}")
    display(df.head(3))